<a href="https://colab.research.google.com/github/mhw47/ML_Practice/blob/main/HousePricePredictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [101]:
import pandas as pd
import os
import sklearn
from ydata_profiling import ProfileReport

/tmp/ipykernel_681/2975017578.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tanyachawla412/house-prices")

print("Path to dataset files:", path)
csv_path = "/kaggle/input/house-prices/House_price.csv"

Using Colab cache for faster access to the 'house-prices' dataset.
Path to dataset files: /kaggle/input/house-prices


In [ ]:
from google.colab import userdata

# Pass your Kaggle credentials to the environment
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')


In [102]:
df = pd.read_csv(csv_path)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4548 entries, 0 to 4547
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Avg. Area Income    4548 non-null   float64
 1   House Age           4548 non-null   float64
 2   Number of Rooms     4548 non-null   float64
 3   Number of Bedrooms  4548 non-null   float64
 4   Area Population     4548 non-null   float64
 5   Price               4548 non-null   float64
 6   Address             4548 non-null   object 
dtypes: float64(6), object(1)
memory usage: 248.8+ KB


In [105]:
profile = ProfileReport(df, title="Exploratory Data Analysis Report")
profile.to_file("data_profile_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 11.88it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
df.shape

(4548, 7)

In [ ]:
df.sample(5)

,Avg. Area Income,House Age,Number of Rooms,Number of Bedrooms,Area Population,Price,Address
3138,77352.63760,9.008900,5.894762,3.49,39667.50726,1898168.684,"969 Robert Vista Suite 545\nWest Kelly, NC 318..."
4010,82919.89104,5.776611,5.900321,3.17,44985.26192,1527676.374,"204 Arias Grove Apt. 165\nBarbarafurt, AZ 6055..."
376,78847.53637,5.573232,7.027355,3.26,20653.89831,1214320.574,530 Anderson Mill Apt. 727\nSouth Brendaboroug...
2659,82713.91148,6.447964,4.583870,4.29,29936.39719,1296223.706,"90307 Mercer Loaf\nJessicaport, MH 44061"
2605,50857.09653,5.512505,8.530697,3.26,40790.18479,1131533.555,USS Brown\nFPO AA 43649-1893


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Separate features (X) and target (Y)
# We drop 'Price' and 'Address' as 'Address' is non-numeric text
X = df.drop(columns=['Price', 'Address'])
Y = df['Price']

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Split plain X and Y (90% study, 10% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1, random_state=42)

# 2. Pick the brain
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)

# 3. Study (.fit)
xgb_model.fit(X_train, Y_train)

# 4. Take the test (.predict)
Y_pred = xgb_model.predict(X_test)

# 5. Grade the test (Directly in real dollars—no multiplying by 1,000,000 needed!)
r2 = r2_score(Y_test, Y_pred)
mae = mean_absolute_error(Y_test, Y_pred)

print(f"R2 Score: {r2:.2%}")
print(f"Average Dollar Error (MAE): ${mae:,.2f}")

R2 Score: 89.01%
Average Dollar Error (MAE): $91,842.84


In [ ]:
custom_sample = pd.DataFrame([{
    'Avg. Area Income': 70000.0,
    'House Age': 6.0,
    'Number of Rooms': 7.0,
    'Number of Bedrooms': 4.0,
    'Area Population': 35000.0
}])

# Predict directly on the custom house without scaling!
predicted_price = xgb_model.predict(custom_sample)[0]
print(f"Predicted House Price: ${predicted_price:,.2f}")

Predicted House Price: $1,253,358.00
